# T-lineage and CAR-T analysis

## Repository navigation

This notebook is a downstream analytical workflow record for T-lineage and CAR-T cohort analyses. It starts from a processed T-lineage/CAR-T AnnData object and records the analysis logic used for metadata harmonization, T-subcluster and CAR-T visualization, cytokine-storm scoring, CAR-T versus endoT comparisons, DEG/enrichment analysis, functional-state scoring, and stage-specific subcluster summaries.

# 1. Load processed T-lineage AnnData object

Environment setup and loading of the processed T-lineage/CAR-T object. This notebook does not perform raw-data preprocessing, alignment, counting, or initial matrix generation.


In [1]:
import os

# Set environment variables to limit the number of threads used by various libraries.
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"
os.environ["VECLIB_MAXIMUM_THREADS"] = "16"
os.environ["NUMEXPR_NUM_THREADS"] = "16"

# Documented private input and output path variables.
# The processed AnnData object is not distributed.
# T_LINEAGE_H5AD and T_LINEAGE_OUTPUT_DIR record the expected environment-variable
# names used in the original downstream workflow.
# Preserve the original launch directory if this setup cell is rerun.
analysis_start_dir = globals().get("analysis_start_dir", os.getcwd())

t_lineage_h5ad_setting = os.environ.get("T_LINEAGE_H5AD", "adata_t.h5ad")
t_lineage_h5ad_path = (
    t_lineage_h5ad_setting
    if os.path.isabs(t_lineage_h5ad_setting)
    else os.path.join(analysis_start_dir, t_lineage_h5ad_setting)
)

t_lineage_output_setting = os.environ.get(
    "T_LINEAGE_OUTPUT_DIR", os.path.join("outputs", "t_lineage")
)
t_lineage_output_dir = (
    t_lineage_output_setting
    if os.path.isabs(t_lineage_output_setting)
    else os.path.join(analysis_start_dir, t_lineage_output_setting)
)
os.makedirs(t_lineage_output_dir, exist_ok=True)


In [2]:
# Dependencies and requirements
# Python kernel used during development: multics (Python 3.11.13)

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors as mcolors

In [3]:
# figure parameters
sc.settings.set_figure_params(frameon=False, dpi=300)
sc.set_figure_params(dpi_save=300)

In [ ]:
# Load and validate the processed T-lineage AnnData object, then direct
# relative outputs to the configured analysis-output directory.
adata_t = sc.read_h5ad(t_lineage_h5ad_path)

In [10]:
if adata_t.n_obs == 0 or adata_t.n_vars == 0:
    raise ValueError('The T-lineage AnnData object must contain cells and genes.')
if not adata_t.obs_names.is_unique:
    raise ValueError('adata_t.obs_names must be unique.')
if not adata_t.var_names.is_unique:
    raise ValueError('adata_t.var_names must be unique.')

required_t_lineage_obs_columns = [
    'sample_id', 'disease', 'stage', 'severity', 't_anno'
]
missing_t_lineage_obs_columns = [
    column for column in required_t_lineage_obs_columns
    if column not in adata_t.obs.columns
]
if missing_t_lineage_obs_columns:
    raise ValueError(
        f"Missing required T-lineage metadata columns: {missing_t_lineage_obs_columns}"
    )

if 'X_umap' not in adata_t.obsm:
    raise ValueError("Missing required embedding: adata_t.obsm['X_umap']")
if adata_t.obsm['X_umap'].shape[0] != adata_t.n_obs:
    raise ValueError("adata_t.obsm['X_umap'] does not match the number of cells.")
if adata_t.obsm['X_umap'].ndim != 2 or adata_t.obsm['X_umap'].shape[1] < 2:
    raise ValueError("adata_t.obsm['X_umap'] must contain at least two dimensions.")

expected_t_lineage_diseases = {'CAR-T_CRS', 'COVID19', 'SLE', 'HC'}
missing_t_lineage_diseases = expected_t_lineage_diseases - set(
    adata_t.obs['disease'].dropna().astype(str)
)
if missing_t_lineage_diseases:
    raise ValueError(
        f"Missing expected T-lineage disease categories: {sorted(missing_t_lineage_diseases)}"
    )

expected_t_lineage_stages = {
    'CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con',
    'COVID19_pro', 'COVID19_con', 'SLE', 'HC'
}
missing_t_lineage_stages = expected_t_lineage_stages - set(
    adata_t.obs['stage'].dropna().astype(str)
)
if missing_t_lineage_stages:
    raise ValueError(
        f"Missing expected T-lineage stage categories: {sorted(missing_t_lineage_stages)}"
    )

expected_t_lineage_severities = {'Moderate', 'Severe', 'No_CRS', 'HC'}
missing_t_lineage_severities = expected_t_lineage_severities - set(
    adata_t.obs['severity'].dropna().astype(str)
)
if missing_t_lineage_severities:
    raise ValueError(
        f"Missing expected T-lineage severity categories: {sorted(missing_t_lineage_severities)}"
    )

if adata_t.obs['sample_id'].isna().any():
    raise ValueError('adata_t.obs contains missing sample_id values.')
if adata_t.obs['t_anno'].isna().any():
    raise ValueError('adata_t.obs contains missing t_anno values.')

os.chdir(t_lineage_output_dir)
sc.settings.figdir = "."
print(f"Loaded T-lineage object: {adata_t.n_obs:,} cells × {adata_t.n_vars:,} genes")

Loaded T-lineage object: 1,290,418 cells × 17,249 genes


# 2. Object inspection and CAR-T identification

The following cells inspect the supplied processed object, construct the combined group label, and identify CAR-T versus endogenous T cells from the recorded sample identifiers. They are not a raw-data preprocessing workflow.


In [8]:
print(adata_t.obs.columns)

Index(['sample_id', 'batch', '_scvi_batch', '_scvi_labels', 'leiden1_anno',
       'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt',
       'pct_counts_mt', 't_anno', 'disease', 'severity', 'stage'],
      dtype='object')


In [17]:
# Create one 'group' column that combines 'stage' and 'severity'
adata_t.obs['group'] = adata_t.obs['stage'].astype(str) + '_' + adata_t.obs['severity'].astype(str)

In [ ]:
# Add a new column 'CAR-T_anno' to separate CAR-T cells and endoT cells
# CAR-T cells are annotated from the source by sample IDs (pure CAR-T cells extracted through FACS)
CAR_T_samples = [
    "CART_S005",
    "CART_S006",
    "CART_S007",
    "CART_S012",
    "CART_S013",
    "CART_S014",
    "CART_S019",
    "CART_S020",
    "CART_S021",
    "CART_S026",
    "CART_S027",
    "CART_S028",
    "CART_S033",
    "CART_S034",
    "CART_S035",
    "CART_S040",
    "CART_S041",
    "CART_S042",
    "CART_S046",
    "CART_S047",
    "CART_S048",
    "CART_S053",
    "CART_S054",
    "CART_S055",
    "CART_S060",
    "CART_S061",
    "CART_S062",
    "CART_S067",
    "CART_S068",
    "CART_S069",
    "CART_S074",
    "CART_S075",
    "CART_S076",
    "CART_S081",
    "CART_S082",
    "CART_S083",
    "CART_S087",
    "CART_S088",
    "CART_S089",
    "CART_S094",
    "CART_S095",
    "CART_S096",
    "CART_S101",
    "CART_S102",
    "CART_S103",
    "CART_S108",
    "CART_S109",
    "CART_S110"
]

# Create the CAR-T_anno column by setting all cells to 'endoT' first
adata_t.obs['CAR-T_anno'] = 'endoT'
# Then set CAR-T samples to 'CAR-T'
adata_t.obs.loc[adata_t.obs['sample_id'].isin(CAR_T_samples), 'CAR-T_anno'] = 'CAR-T'

# Validate the constructed annotation before downstream comparisons
expected_car_t_annotations = {'CAR-T', 'endoT'}
observed_car_t_annotations = set(adata_t.obs['CAR-T_anno'].dropna().astype(str))
if observed_car_t_annotations != expected_car_t_annotations:
    raise ValueError(
        'CAR-T_anno must contain exactly CAR-T and endoT; observed: '
        f"{sorted(observed_car_t_annotations)}"
    )
if not adata_t.obs.loc[adata_t.obs['CAR-T_anno'] == 'CAR-T', 'sample_id'].isin(CAR_T_samples).all():
    raise ValueError('CAR-T_anno contains CAR-T cells outside the approved CAR_T_samples mapping.')

print("Number of cells in each CAR-T_anno category:")
print(adata_t.obs['CAR-T_anno'].value_counts())

Number of cells in each CAR-T_anno category:
CAR-T_anno
endoT    1016154
CAR-T     274264
Name: count, dtype: int64


# 3. T-cell subcluster and CAR-T identification UMAP
---
Fig. S3A, Fig. S3B

In [ ]:
# T cell subcluster UMAP (Fig. S3A)

plt.rcParams['figure.dpi'] = 300

# Major T subcluster UMAP
t_palette = {
    # CD4+ T cells
    'Tn_CD4_LEF1':     "#5098CF",  
    'Tcm_CD4_IL7R':    "#EB7952",  
    'Tem_CD4_GZMK':    "#53BDA8",
    'Treg_CD4_FOXP3':   "#E74D35",  
    'Ta_CD4_MKI67':"#C864E0",

    # CD8+ T cells
    'Tn_CD8_LEF1':    "#B8A07D",  
    'Tem_CD8_GZMK':    "#F28E8E",
    'Tctl_CD8_GZMB':   "#0C883B",
    'Ta_CD8_MKI67':"#2F6690",
    'Tex_CD8_PDCD1':    "#A23450", 

    # Tnkl_KLRD1 cells
    'Tnkl_KLRD1':        "#F4A261"
}

# plot UMAP
sc.pl.umap(
    adata_t,
    color = 't_anno',
    palette = t_palette,
    frameon = False,
    size=0.5,
    save='_t.png'
)

In [ ]:
# UMAP by 'CAR-T_anno'; cells other than CAR-T are grey (Fig. S3B)

# Create figure
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

#  Plot background (all cells in lightgrey)

sc.pl.umap(
    adata_t,
    color='CAR-T_anno',
    palette={cat: 'lightgrey' for cat in adata_t.obs['CAR-T_anno'].unique()},
    size=1,
    frameon=False,
    ax=ax,
    legend_loc=None,
    show=False
)

# Overlay CAR-T cells
adata_cart = adata_t[adata_t.obs['CAR-T_anno'] == 'CAR-T']

sc.pl.umap(
    adata_cart,
    color='CAR-T_anno',
    palette={'CAR-T': "#3177a3"},
    size=1,
    frameon=False,
    ax=ax,
    title='CAR-T Cells',
    legend_loc='none',
    show=False
)

plt.tight_layout()
plt.savefig('UMAP_CAR-T.png', bbox_inches='tight', dpi=300)
plt.show()

# 4. T-cell and CAR-T CS scoring

## Calculate T cell-specific CS score

In [14]:
# T cell specific CS gene list
cs_genes = [
    # Classical cytokines by T cells
    'TNF', 'IFNG', 'IL2', 'CSF2',
    # Key chemokines
    'CCL3', 'CCL4', 'CCL5',
    # Cytotoxicity/ligands
    'FASLG', 'CD40LG', 'TNFSF13B', 'GZMB', 'GZMK', 'PRF1',
    # Activation (receptors)
    'IL2RA', 'TNFRSF9', 'TNFRSF4', 'ICOS',
    # Cytokine receptors
    'IL6R', 'IL1R1', 'IL6ST', 
    # Checkpoints
    'PDCD1', 'LAG3', 'CTLA4', 'HAVCR2',
    # Signaling
    'STAT1', 'STAT3', 'STAT5A', 'NFKBIA', 'RELA', 'TNFAIP3'
]

In [15]:
# Calculate
sc.tl.score_genes(
    adata_t, 
    gene_list=cs_genes, 
    score_name='CS_score', 
    use_raw=False,
    ctrl_size=len(cs_genes)
)

## T cell CS score across groups
---
Table S4a, Fig. 3A

In [18]:
# generate per sample CS scores of groups of interest
# Metadata for Fig. 3A, Table S4a
groups_of_interest = ['CAR-T_CRS_pro_Moderate', 'CAR-T_CRS_pro_Severe', 'COVID19_pro_Severe', 'COVID19_pro_Moderate', 'SLE_Severe', 'SLE_Moderate', 'HC_HC']
# Filter the data
mask = adata_t.obs['group'].isin(groups_of_interest)
filtered_data = adata_t.obs[mask].copy()

# Calculate mean CS score per sample
# grouping by both sample_id and group to keep the group label
sample_cs_scores = filtered_data.groupby(['sample_id', 'group'], observed=True)['CS_score'].mean().reset_index()

# Sort the results by group for better organization
# To respect the order in groups_of_interest, we can make 'group' categorical
sample_cs_scores['group'] = pd.Categorical(sample_cs_scores['group'], categories=groups_of_interest, ordered=True)
sample_cs_scores = sample_cs_scores.sort_values('group')

# Save to Excel (Table S4a)
sample_cs_scores.to_excel('CS_score_per_sample_selected_groups.xlsx', index=False)

## CAR-T vs endoT CS score
---
Table S4b, Fig. 3B

In [ ]:
# subset adata_t to only CAR-T_CRS groups
adata_cart_crs = adata_t[adata_t.obs['disease'] == 'CAR-T_CRS'].copy()

# Remove No_CRS samples for comparison across CRS stages
adata_cart_crs = adata_cart_crs[adata_cart_crs.obs['severity'] != 'No_CRS'].copy()

# Remove samples without their pairs
no_pair_samples = [
        "CART_S001",
        "CART_S008",
        "CART_S015",
        "CART_S022",
        "CART_S029",
        "CART_S036",
        "CART_S043",
        "CART_S049",
        "CART_S056",
        "CART_S063",
        "CART_S070",
        "CART_S077",
        "CART_S084",
        "CART_S090",
        "CART_S097",
        "CART_S104",
        'CART_S087',
        'CART_S094',
        'CART_S047']

# Exclude unpaired samples
adata_cart_crs = adata_cart_crs[~adata_cart_crs.obs['sample_id'].isin(no_pair_samples)].copy()

In [20]:
# Calculate mean CS score per CAR-T_anno x sample_id
cart_cs_scores = (
    adata_cart_crs[adata_cart_crs.obs['CAR-T_anno'] == 'CAR-T']
    .obs.groupby(['sample_id'], observed=True)['CS_score']
    .mean()
    .dropna()
    .reset_index()
)

endoT_cs_scores = (
    adata_cart_crs[adata_cart_crs.obs['CAR-T_anno'] == 'endoT']
    .obs.groupby(['sample_id'], observed=True)['CS_score']
    .mean()
    .dropna()
    .reset_index()
)

# Function to create a common paired key between endoT (S2/S3/S4) and CAR-T (S5/S6/S7)
def get_pair_key(s):
    s = str(s)
    if s.endswith('S2') or s.endswith('S5'):
        return s[:-2] + 'T1'
    elif s.endswith('S3') or s.endswith('S6'):
        return s[:-2] + 'T2'
    elif s.endswith('S4') or s.endswith('S7'):
        return s[:-2] + 'T3'
    return s

# Apply mapping to create the pair_key column
cart_cs_scores['pair_key'] = cart_cs_scores['sample_id'].apply(get_pair_key)
endoT_cs_scores['pair_key'] = endoT_cs_scores['sample_id'].apply(get_pair_key)

# Merge the CAR-T and endoT scores by pair_key (drop rows lacking parallel samples if any)
matched_scores = pd.merge(
    cart_cs_scores,
    endoT_cs_scores,
    on=['pair_key'],
    suffixes=('_CAR-T', '_endoT')
)

# Save to Excel (Table S4b)
matched_scores.to_excel('CS_score_CAR-T_vs_endoT_matched.xlsx', index=False)

## CAR-T vs endoT CS gene expression across stages
---
Fig. 3D

In [ ]:
# Heatmap of CS-associated gene expression in CAR-T and endogenous T cells across CAR-T CRS stages
# use adata_cart_crs, which is already filtered to only CAR-T CRS samples and excludes No_CRS and unpaired samples

cs_genes = [
    # Classical cytokines by T cells
    'TNF', 'IFNG', 'IL2', 'CSF2',
    # Key chemokines
    'CCL3', 'CCL4', 'CCL5',
    # Cytotoxicity/ligands
    'FASLG', 'CD40LG', 'TNFSF13B', 'GZMB', 'GZMK', 'PRF1',
    # Activation receptors
    'IL2RA', 'TNFRSF9', 'TNFRSF4', 'ICOS',
    # Cytokine receptors
    'IL6R', 'IL1R1', 'IL6ST', 
    # Checkpoints
    'PDCD1', 'LAG3', 'CTLA4', 'HAVCR2',
    # Signaling regulators
    'STAT1', 'STAT3', 'STAT5A', 'NFKBIA', 'RELA', 'TNFAIP3'
]

# avoid modifying a potential AnnData view
adata_cart_crs = adata_cart_crs.copy()

# Create heatmap grouping variable
adata_cart_crs.obs['heatmap_group'] = (
    adata_cart_crs.obs['CAR-T_anno'].astype(str) + '_' +
    adata_cart_crs.obs['stage'].astype(str)
)

# Validate the complete, final CS-associated gene set before plotting
missing_cs_genes = [gene for gene in cs_genes if gene not in adata_cart_crs.var_names]
if missing_cs_genes:
    raise ValueError(f"Missing CS-associated genes: {missing_cs_genes}")

# Extract expression values from adata_cart_crs.X
exp_df = sc.get.obs_df(
    adata_cart_crs,
    keys=cs_genes + ['heatmap_group'],
    use_raw=False
)

# Calculate mean expression per heatmap group
avg_exp = exp_df.groupby('heatmap_group')[cs_genes].mean().T

# Define intended column order
stage_order = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']
cart_order = ['CAR-T', 'endoT']

sorted_columns = []
for c_type in cart_order:
    for stage in stage_order:
        col_name = f"{c_type}_{stage}"
        if col_name in avg_exp.columns:
            sorted_columns.append(col_name)

print("Columns found for plotting:", sorted_columns)

# Reorder columns
avg_exp = avg_exp[sorted_columns]

# Remove genes with zero variance across plotted groups;
avg_exp = avg_exp.loc[avg_exp.std(axis=1) > 0]

# Plot clustermap
# z_score=0 standardizes rows, so each gene is displayed as relative expression across groups
g = sns.clustermap(
    avg_exp,
    cmap='RdBu_r',
    z_score=0,
    col_cluster=False,
    row_cluster=True,
    xticklabels=True,
    yticklabels=True,
    figsize=(6, len(avg_exp) * 0.35 + 2),
    center=0,
    cbar_kws={'label': 'Z-score expression'}
)

# Adjust label appearance
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    rotation=90,
    fontsize=8
)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    fontsize=8
)

g.ax_heatmap.set_xlabel('Group')
g.ax_heatmap.set_ylabel('CS-associated genes')

plt.savefig('CS_genes_clustermap_CAR-T_endoT_across_CRS_stages.pdf', bbox_inches='tight')
plt.show()

## CAR-T subcluster dynamics and CS score
---
Table S4e, Fig. S3H

In [22]:
# Use adata_cart_crs (restricted to CAR-T_CRS, removed No_CRS and non-paired samples) from the previous section

# Subset to only CAR-T cells
adata_cart = adata_cart_crs[adata_cart_crs.obs['CAR-T_anno'] == 'CAR-T'].copy()

In [ ]:
# Prepare the data
# Calculate cell counts and mean CS score for each (t_anno, stage) pair
plot_data = (
    adata_cart.obs.groupby(['t_anno', 'stage'], observed=True)
    .agg(
        cell_count=('sample_id', 'count'),          # Dot size: Number of cells
        mean_cs_score=('CS_score', 'mean')   # Dot color: Mean scaled CS score
    )
    .reset_index()
)

# Filter for the specific timepoints
time_order = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']
plot_data = plot_data[plot_data['stage'].isin(time_order)].copy()

# handle -inf and +inf values in mean_cs_score
# Replace -inf with the minimum finite value in the column
min_finite_score = plot_data.loc[np.isfinite(plot_data['mean_cs_score']), 'mean_cs_score'].min()
plot_data['mean_cs_score'] = plot_data['mean_cs_score'].replace(-np.inf, min_finite_score)
# handle +inf if it exists
max_finite_score = plot_data.loc[np.isfinite(plot_data['mean_cs_score']), 'mean_cs_score'].max()
plot_data['mean_cs_score'] = plot_data['mean_cs_score'].replace(np.inf, max_finite_score)

# Set categorical order for axes
plot_data['stage'] = pd.Categorical(plot_data['stage'], categories=time_order, ordered=True)
plot_data['x_code'] = plot_data['stage'].cat.codes

# Normalize size for plotting
min_size = 20
max_size = 500
min_count = plot_data['cell_count'].min()
max_count = plot_data['cell_count'].max()

if max_count > min_count:
    plot_data['plot_size'] = (
        (plot_data['cell_count'] - min_count) / (max_count - min_count)
    ) * (max_size - min_size) + min_size
else:
    plot_data['plot_size'] = (min_size + max_size) / 2


# Plotting
fig, ax = plt.subplots(figsize=(12, 6), dpi=300)

scatter = ax.scatter(
    x=plot_data['x_code'],
    y=plot_data['t_anno'],
    s=plot_data['plot_size'],
    c=plot_data['mean_cs_score'],
    cmap='RdBu_r', 
    linewidth=0.5,
    alpha=0.9
)

cbar = plt.colorbar(scatter, ax=ax, fraction=0.05, pad=0.04)
cbar.set_label('Mean CS Score')

# dot size legend
sizes_legend = [int(min_count), int((min_count + max_count)/2), int(max_count)]
legend_handles = []
for s in sizes_legend:
    if max_count > min_count:
        scaled_s = ((s - min_count) / (max_count - min_count)) * (max_size - min_size) + min_size
    else:
        scaled_s = (min_size + max_size) / 2
    legend_handles.append(
        plt.scatter([], [], s=scaled_s, c='gray', edgecolors='black', label=str(s))
    )

legend = ax.legend(
    handles=legend_handles, 
    title='Cell Count', 
    bbox_to_anchor=(1.2, 1), 
    loc='upper left',
    frameon=False,
    labelspacing=1.5
)

# layout
ax.set_xticks(range(len(time_order)))
ax.set_xticklabels(time_order, rotation=45, ha='right')

ax.set_xlabel('')
ax.set_ylabel('T Cell Subcluster')
ax.set_title('CAR-T Subcluster Dynamics & CS Score', fontsize=10)
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_axisbelow(True) 

plt.tight_layout()
plt.savefig('dotplot_CAR-T_subcluster_dynamics_CS_score.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Export plotting matrix to Excel (Table S4e)
output_xlsx = "dotplot_CAR-T_subcluster_dynamics_plotting_matrix.xlsx"

# Make cell-count matrix
cell_count_matrix = (
    plot_data
    .pivot(index="t_anno", columns="stage", values="cell_count")
    .reindex(columns=time_order)
    .reset_index()
    .rename(columns={"t_anno": "subcluster"})
)

# Make mean CS-score matrix
cs_score_matrix = (
    plot_data
    .pivot(index="t_anno", columns="stage", values="mean_cs_score")
    .reindex(columns=time_order)
    .reset_index()
    .rename(columns={"t_anno": "subcluster"})
)
# Export to xlsx
with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    cell_count_matrix.to_excel(
        writer,
        sheet_name="cell_count",
        index=False
    )
    
    cs_score_matrix.to_excel(
        writer,
        sheet_name="mean_CS_score",
        index=False
    )

# 5. CAR-T DEG and enrichment analyses

In [ ]:
# Restrict to CAR-T cells
adata_cart = adata_t[adata_t.obs['CAR-T_anno'] == 'CAR-T'].copy()

## DEGs: CRS_vs_No_CRS
---
Table S4d, Fig. S3D

In [ ]:
# Calculate DEGs for CAR-T cells: (Moderate + Severe) vs No_CRS

# Create a comparison group column
# Map 'Moderate' and 'Severe' to 'CRS', and keep 'No_CRS' as is
adata_cart.obs['crs_comparison'] = adata_cart.obs['severity'].replace({
    'Moderate': 'CRS',
    'Severe': 'CRS'
})

# Perform Differential Expression Analysis (Rank Genes Groups)
# Group 'CRS' (Moderate+Severe) vs Reference 'No_CRS'
print("Calculating DEGs (CRS vs No_CRS)...")
sc.tl.rank_genes_groups(
    adata_cart,
    groupby='crs_comparison',
    reference='No_CRS',
    groups=['CRS'],
    method='wilcoxon',
    use_raw=False
)

# Extract and Save Results
cart_crs_deg_df = sc.get.rank_genes_groups_df(adata_cart, group='CRS')

# Save to CSV (Table S4d)
cart_crs_deg_df.to_csv('DEG_CAR-T_CRS_vs_NoCRS.csv', index=False)

In [ ]:
# Functional enrichment

import gseapy as gp

# filter for significant upregulated DEGs (padj < 0.05, logFC > 0.5)
sig_up_genes = cart_crs_deg_df[
    (cart_crs_deg_df['pvals_adj'] < 0.05) &
    (cart_crs_deg_df['logfoldchanges'] > 0.5)
]['names'].tolist()

# Run GO
cart_crs_go_up = gp.enrichr(
    gene_list=sig_up_genes,
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir=None
)

In [ ]:
# Barplot (Fig. S3D)
from matplotlib import colors as mcolors

# output results as df
cart_crs_go_complete_df = cart_crs_go_up.results.copy()
cart_crs_go_complete_df.to_excel('CAR-T_CRS_vs_No_CRS_GO_enrichment_complete.xlsx', index=False)
cart_crs_go_up_df = cart_crs_go_complete_df.copy()

# filter for significant terms (adjusted p-value < 0.05)
cart_crs_go_up_df = cart_crs_go_up_df[cart_crs_go_up_df['Adjusted P-value'] < 0.05]

# sort genes in 'Genes' column by logFC
gene_rank_dict = dict(zip(cart_crs_deg_df['names'], cart_crs_deg_df['logfoldchanges']))

def sort_genes_by_lfc(gene_string):
    if pd.isna(gene_string): return gene_string
    genes = gene_string.split(';')
    genes_sorted = sorted(genes, key=lambda x: gene_rank_dict.get(x, 0), reverse=True)
    return ';'.join(genes_sorted)

cart_crs_go_up_df['Genes'] = cart_crs_go_up_df['Genes'].apply(sort_genes_by_lfc)

# terms of interest
key_terms = [
    'Positive Regulation Of Cell Cycle Process (GO:0090068)',
    'Glycolytic Process (GO:0006096)',
    'Positive Regulation Of Cytokine Production (GO:0001819)',
    'Positive Regulation Of T Cell Proliferation (GO:0042102)',
    'Positive Regulation Of Monocyte Chemotaxis (GO:0090026)'
]

# Filter for selected terms
cart_crs_go_up_df = cart_crs_go_up_df[cart_crs_go_up_df['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

cart_crs_go_up_df['Overlap_decimal'] = cart_crs_go_up_df['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
cart_crs_go_up_df = cart_crs_go_up_df.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
cart_crs_go_up_df['neg_log_padj'] = -np.log10(cart_crs_go_up_df['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('OrRd')
norm = mcolors.Normalize(vmin=cart_crs_go_up_df['neg_log_padj'].min(), vmax=cart_crs_go_up_df['neg_log_padj'].max())
colors = cmap(norm(cart_crs_go_up_df['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=cart_crs_go_up_df['Term'],
    width=cart_crs_go_up_df['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('CAR-T_CRS_GO.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

## DEGs: CAR-T cells, Moderate vs Severe
---
Table S4d, Fig. S3D

In [ ]:
# Calculate DEGs for CAR-T cells: Severe vs Moderate

# exclude 'No_CRS'
adata_cart_sev = adata_cart[adata_cart.obs['severity'].isin(['Moderate', 'Severe'])].copy()

# Restrict to CAR-T_CRS_pro stage for comparison
adata_cart_sev = adata_cart_sev[adata_cart_sev.obs['stage'] == 'CAR-T_CRS_pro'].copy()

# DEG analysis: Severe vs Moderate
sc.tl.rank_genes_groups(
    adata_cart_sev,
    groupby='severity',
    reference='Moderate',
    groups=['Severe'],
    method='wilcoxon',
    use_raw=False
)

# Extract and Save Results
cart_sev_deg_df = sc.get.rank_genes_groups_df(adata_cart_sev, group='Severe')

# Save to CSV (Table S4d)
cart_sev_deg_df.to_csv('DEG_CAR-T_Severe_vs_Moderate.csv', index=False)

In [27]:
import gseapy as gp

# filter for significant upregulated DEGs (padj < 0.05, logFC > 0.5)
sig_up_genes = cart_sev_deg_df[
    (cart_sev_deg_df['pvals_adj'] < 0.05) &
    (cart_sev_deg_df['logfoldchanges'] > 0.5)
]['names'].tolist()

# Run GO
cart_sev_go_up = gp.enrichr(
    gene_list=sig_up_genes,
    gene_sets=['GO_Biological_Process_2023'],
    organism='Human',
    outdir=None
)

In [ ]:
# Barplot (Fig. S3D)

# output results as df
cart_sev_go_up_df = cart_sev_go_up.results

# filter for significant terms (adjusted p-value < 0.05)
cart_sev_go_up_df = cart_sev_go_up_df[cart_sev_go_up_df['Adjusted P-value'] < 0.05]

# sort genes in 'Genes' column by logFC
gene_rank_dict = dict(zip(cart_sev_deg_df['names'], cart_sev_deg_df['logfoldchanges']))

def sort_genes_by_lfc(gene_string):
    if pd.isna(gene_string): return gene_string
    genes = gene_string.split(';')
    genes_sorted = sorted(genes, key=lambda x: gene_rank_dict.get(x, 0), reverse=True)
    return ';'.join(genes_sorted)

cart_sev_go_up_df['Genes'] = cart_sev_go_up_df['Genes'].apply(sort_genes_by_lfc)

# Save the enrichment table used by the following visualization.
cart_sev_go_up_df.to_excel('CAR-T_Severe_vs_Moderate_GO.xlsx', index=False)
cart_sev_go_df = pd.read_excel('CAR-T_Severe_vs_Moderate_GO.xlsx')

# terms of interest
key_terms = [
    'Positive Regulation Of Monocyte Chemotaxis (GO:0090026)',
    'Positive Regulation Of Cytokine Production (GO:0001819)',
    'T Cell Chemotaxis (GO:0010818)',
    'Chemokine-Mediated Signaling Pathway (GO:0070098)',
    'Positive Regulation Of Tumor Necrosis Factor Superfamily Cytokine Production (GO:1903557)'
]

# Filter for selected terms
cart_sev_go_df = cart_sev_go_df[cart_sev_go_df['Term'].isin(key_terms)].copy()

# Convert Overlap from "num/den" to decimal
def parse_overlap(fraction_str):
    num, den = fraction_str.split('/')
    return float(num) / float(den)

cart_sev_go_df['Overlap_decimal'] = cart_sev_go_df['Overlap'].apply(parse_overlap)

# sort by Overlap_decimal
cart_sev_go_df = cart_sev_go_df.sort_values('Overlap_decimal', ascending=True)

# Calculate -log10(Adjusted P-value) for coloring
cart_sev_go_df['neg_log_padj'] = -np.log10(cart_sev_go_df['Adjusted P-value'])

# Define OrRd colormap and normalize based on the -log10 values
cmap = plt.get_cmap('OrRd')
norm = mcolors.Normalize(vmin=cart_sev_go_df['neg_log_padj'].min(), vmax=cart_sev_go_df['neg_log_padj'].max())
colors = cmap(norm(cart_sev_go_df['neg_log_padj']))

# plot
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.barh(
    y=cart_sev_go_df['Term'],
    width=cart_sev_go_df['Overlap_decimal'],
    color=colors
)

# Add a colorbar to show the scale of Adjusted P-values
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('-log10(Adjusted P-value)')

plt.xlabel('Overlap')
plt.tight_layout()
plt.savefig('CAR-T_Severe_vs_Moderate_GO.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

## Volcano plot for CRS_vs_No_CRS and S_vs_M
---
Fig. S3C

In [ ]:
# Bidirectional DEG volcano plot 
# Input files: DEG_CAR-T_CRS_vs_NoCRS.csv and DEG_CAR-T_Severe_vs_Moderate.csv

from adjustText import adjust_text

# read data
df1 = pd.read_csv('DEG_CAR-T_CRS_vs_NoCRS.csv').assign(group='CRS_vs_NoCRS')
df2 = pd.read_csv('DEG_CAR-T_Severe_vs_Moderate.csv').assign(group='Severe_vs_Moderate')
df = pd.concat([df1, df2], ignore_index=True)

# filter for pvals_adj < 0.05
df = df[df['pvals_adj'] < 0.05].copy()
# Filter for logfoldchanges magnitude > 0.5 (UP > 0.5, DOWN < -0.5)
df = df[np.abs(df['logfoldchanges']) > 0.5].copy()

# gene labels for the volcano plot
label = {
    'CRS_vs_NoCRS': [
        # up
        'MX1', 'IL2RA', 'TNFSF10', 'CCR7', 'LTA', 'IFI44L', 'TNF', 'IFIT1', 'IFI27', 'CD86', 'IL1R2', 
        # down
        'JUND', 'NFKBIA', 'DUSP1', 'TNFAIP3', 'CD69', 'GZMK', 'PDCD2', 'NFKBIZ', 'CCL23', 'DUSP2'],
    'Severe_vs_Moderate': [
        # UP
        'GZMA', 'NKG7', 'GZMK', 'GZMB', 'PRF1', 'IFNG', 'CD86', 'CSF2', 'CCR2', 'PDCD1', 'LAG3',
        # DOWN
        'IFIT1', 'KIR3DL1', 'IFI44L', 'IFIT3', 'IFI44', 'CXCL16']
}

# point size
df['point_size'] = np.log1p(np.abs(df['scores']))

# assign labels
df['label'] = df.apply(
    lambda r: r['names'] if r['names'] in label[r['group']] else None,
    axis=1
)

# group order
group_order = ['CRS_vs_NoCRS', 'Severe_vs_Moderate']
x_map = {g: i for i, g in enumerate(group_order)}
df['x'] = df['group'].map(x_map)

# grey background boxes
shades = (
    df.groupby('group')['logfoldchanges']
    .agg(['min', 'max'])
    .reset_index()
)
shades['x'] = shades['group'].map(x_map)

# colors
df['color'] = np.where(
    df['label'].notna(),
    np.where(df['logfoldchanges'] > 0, 'firebrick', 'steelblue'),
    "#b3b1b1ff"
)

# jitter
rng=np.random.default_rng(seed=1)
df['x_jitter'] = df['x'] + rng.uniform(-0.3, 0.3, size=len(df))

# Sort dataframe so labeled (colored) points are plotted on top
df['is_labeled'] = df['label'].notna()
df = df.sort_values('is_labeled', ascending=True)

# plotting
fig, ax = plt.subplots(figsize=(8,12))

# plot grey backgrounds based on data range
for _, r in shades.iterrows():
    # Upregulated region background
    if r['max'] > 0.5:
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, 0.5), 
                width=0.7,
                height=r['max'] - 0.5 + 0.5,
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )
    # Downregulated region background
    if r['min'] < -0.5:
        y_start = r['min'] - 0.5
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, y_start),
                width=0.7,
                height=-0.5 - y_start, # Height from bottom to threshold -0.5
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )

# scatter points
ax.scatter(
    df['x_jitter'],
    df['logfoldchanges'],
    s=df['point_size'] * 50,
    c=df['color'],
    alpha=0.8,
    zorder=1
)
# gene labels
texts=[]
for _, r in df.dropna(subset=['label']).iterrows():
    texts.append(
        ax.text(
            r['x_jitter'],
            r['logfoldchanges'],
            r['label'],
            fontsize=8,
            color='black'
        )
    )
adjust_text(
    texts,
    ax=ax,
    only_move={'points':'y', 'texts':'y'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8),
    lim=200
)

# axes
ax.set_xlim(-0.6, len(group_order) - 0.4)

# Set dynamic symmetric y limits
y_abs_max = max(abs(df['logfoldchanges'].min()), abs(df['logfoldchanges'].max())) + 1
ax.set_ylim(-y_abs_max, y_abs_max)

# Add group labels on x axis
ax.set_xticks(list(x_map.values()))
ax.set_xticklabels(list(x_map.keys()))
ax.set_xlabel('')
ax.set_ylabel('logfoldchanges')

# Add 0 line
ax.axhline(0, color='gray', linestyle='--', lw=0.8)

for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis='x', length=0)

plt.tight_layout()
plt.savefig('CRS_vs_Severe_vs_Moderate_DEGs.pdf', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

## DEGs: CRS vs No CRS at before stage
---
Table S4d, Fig. 3E

In [ ]:
# Subset to only CAR-T_CRS_before
adata_t_before = adata_cart[adata_cart.obs['stage'] == 'CAR-T_CRS_before'].copy()

# Create a mapping function that maps No_CRS as 'No_CRS', and Moderate/Severe as 'CRS'
def map_severity(severity):
    if severity == 'No_CRS':
        return 'No_CRS'
    elif severity in ['Moderate', 'Severe']:
        return 'CRS'
    else:
        return None  # For any unexpected values
# apply
adata_t_before.obs['crs'] = adata_t_before.obs['severity'].apply(map_severity)

In [ ]:
# Perform DE analysis between CRS and No_CRS in CAR-T_CRS_before cells
sc.tl.rank_genes_groups(
    adata_t_before,
    groupby='crs',
    reference='No_CRS',
    groups=['CRS'],
    method='wilcoxon',
    use_raw=False
)

# Extract results
before_deg_df = sc.get.rank_genes_groups_df(adata_t_before, group='CRS')

# filter for significant genes (padj < 0.05)
sig_before_deg_df = before_deg_df[before_deg_df['pvals_adj'] < 0.05].copy()

# save (Table S4d)
sig_before_deg_df.to_csv('deg_before_CRS_vs_NoCRS.csv', index=False)

In [ ]:
# monodirectional DEG plot for key DEGs of deg_before_CRS_vs_NoCRS.csv

from adjustText import adjust_text

# Filter for logfoldchanges magnitude > 0.5 (UP > 0.5, DOWN < -0.5)
df = df[np.abs(df['logfoldchanges']) > 0.5].copy()


# ---label---
label = {
    'CRS_vs_NoCRS': [
        # up
        'CSF2', 'IL1R2', 'IL2RA', 'CD40LG', 'LTA', 'IFNG', 'ICAM1', 'CXCL10', 'TNF',
        # down
        'IRS2'
    ]
}

# ---point size---
df['point_size'] = np.log1p(np.abs(df['scores']))

# ---keep only the single group---
group_order = ['CRS_vs_NoCRS']
if 'group' not in df.columns:
    df['group'] = group_order[0]

df = df[df['group'].isin(group_order)].copy()

# ---assign labels---
df['label'] = df.apply(
    lambda r: r['names'] if r['names'] in label[r['group']] else None,
    axis=1
)

# ---group order---
x_map = {g: i for i, g in enumerate(group_order)}
df['x'] = df['group'].map(x_map)

# ---grey background ranges---
shades = (
    df.groupby('group')['logfoldchanges']
    .agg(['min', 'max'])
    .reset_index()
)
shades['x'] = shades['group'].map(x_map)

# ---colors---
df['color'] = np.where(
    df['label'].notna(),
    np.where(df['logfoldchanges'] > 0, 'firebrick', 'steelblue'),
    "#b3b1b1ff"
)

# ---jitter---
rng = np.random.default_rng(seed=1)
df['x_jitter'] = df['x'] + rng.uniform(-0.3, 0.3, size=len(df))

# Plot unlabeled first, labeled last
df['is_labeled'] = df['label'].notna()
df = df.sort_values('is_labeled', ascending=True)

# ---plot---
fig, ax = plt.subplots(figsize=(6, 10))

for _, r in shades.iterrows():
    if r['max'] > 0.5:
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, 0.5),
                width=0.7,
                height=r['max'] - 0.5 + 0.5,
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )
    if r['min'] < -0.5:
        y_start = r['min'] - 0.5
        ax.add_patch(
            plt.Rectangle(
                (r["x"] - 0.35, y_start),
                width=0.7,
                height=-0.5 - y_start,
                color='#ededed',
                alpha=0.5,
                zorder=0
            )
        )

ax.scatter(
    df['x_jitter'],
    df['logfoldchanges'],
    s=df['point_size'] * 50,
    c=df['color'],
    alpha=0.8,
    zorder=1
)

texts = []
for _, r in df.dropna(subset=['label']).iterrows():
    texts.append(
        ax.text(
            r['x_jitter'],
            r['logfoldchanges'],
            r['label'],
            fontsize=8,
            color='black'
        )
    )

adjust_text(
    texts,
    ax=ax,
    only_move={'points': 'y', 'texts': 'y'},
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8),
    lim=200
)

# axes
ax.set_xlim(-0.6, len(group_order) - 0.4)

y_abs_max = max(abs(df['logfoldchanges'].min()), abs(df['logfoldchanges'].max())) + 1
ax.set_ylim(-y_abs_max, y_abs_max)

ax.set_xticks(list(x_map.values()))
ax.set_xticklabels(list(x_map.keys()))
ax.set_xlabel('')
ax.set_ylabel('logfoldchanges')

ax.axhline(0, color='gray', linestyle='--', lw=0.8)

for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis='x', length=0)

plt.tight_layout()
plt.savefig('CRS_NoCRS_DEGs.pdf', dpi=300, bbox_inches='tight')
plt.show()


# 6. CAR-T and endoT subcluster dynamics
---
Fig. S3F

In [40]:
# Prepare metadata for comparing T-cell-subcluster proportions between CAR-T and endoT cells

adata_cart_crs = adata_t[adata_t.obs['disease'] == 'CAR-T_CRS'].copy()

# remove No_CRS samples
adata_cart_crs = adata_cart_crs[adata_cart_crs.obs['severity'] != 'No_CRS'].copy()

meta_counts = adata_cart_crs.obs[['CAR-T_anno', 't_anno', 'stage', 'sample_id']].copy()
meta_counts = (
    meta_counts
    .groupby(['CAR-T_anno', 't_anno', 'stage', 'sample_id'], observed=True)
    .size()
    .reset_index(name='n_cells')
)

# Total cells per sample
sample_totals = (
    adata_cart_crs.obs.groupby(['stage', 'sample_id'], observed=True)
    .size()
    .reset_index(name='total_cells')
)

# Compute fraction per subcluster
meta_counts = meta_counts.merge(sample_totals, on=['stage', 'sample_id'])
meta_counts['fraction'] = meta_counts['n_cells'] / meta_counts['total_cells']

In [ ]:
# Diverging barplot of fractions between CAR-T and endoT cells across groups
groups = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']

for group in groups:
    # Filter out extreme outliers in 'fraction' (both below 1st and above 99th percentile)
    meta_group = meta_counts[meta_counts['stage'] == group].copy()
    
    fraction_low = meta_group['fraction'].quantile(0.01)
    fraction_high = meta_group['fraction'].quantile(0.99)
    meta_group = meta_group[(meta_group['fraction'] >= fraction_low) & (meta_group['fraction'] <= fraction_high)]
    
    # Aggregate fractions per subcluster and cell type (mean across samples)
    fra_group_agg = (
        meta_group
        .groupby(['t_anno', 'CAR-T_anno'])['fraction']
        .mean()
        .unstack()
        .reset_index()
    )
    
    # convert fraction values of endoT to negative for diverging barplot
    fra_group_agg['endoT'] = -fra_group_agg['endoT']
    # sort the subclusters based on CAR-T fraction
    fra_group_plot = fra_group_agg.sort_values(by='CAR-T')
    
    # set the plot y axis range
    y = np.arange(len(fra_group_plot))
    plt.figure(figsize=(6,4), dpi=300)
    # left bar for endoT
    plt.barh(y, fra_group_plot['endoT'], color='#2F6690', label='endoT')
    # right bar for CAR-T
    plt.barh(y, fra_group_plot['CAR-T'], color='#EB7952', label='CAR-T')
    # y-tick labels
    plt.yticks(y, fra_group_plot['t_anno'])
    # vertical line at x=0
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Fraction of cells')
    plt.title(f'Fractions in {group}')
    plt.legend()
    plt.tight_layout()
    
    # Save figure
    plt.savefig(f'Fra_CAR-T_vs_endoT_{group}.pdf', dpi=300, bbox_inches='tight')
    
    plt.show()

# 7. CAR-T versus endoT functional-state scoring

## Data preparation

In [ ]:
# Subset to CAR-T_CRS only
adata_cart_crs = adata_t[adata_t.obs['disease'] == 'CAR-T_CRS']

# Remove No_CRS samples
adata_cart_crs = adata_cart_crs[adata_cart_crs.obs['severity'] != 'No_CRS'].copy()

# Remove unpaired samples
removed_samples = [
        "CART_S001",
        "CART_S008",
        "CART_S015",
        "CART_S022",
        "CART_S029",
        "CART_S036",
        "CART_S043",
        "CART_S049",
        "CART_S056",
        "CART_S063",
        "CART_S070",
        "CART_S077",
        "CART_S084",
        "CART_S090",
        "CART_S097",
        "CART_S104",
        'CART_S087',
        'CART_S094',
        'CART_S047']

adata_cart_crs = adata_cart_crs[~adata_cart_crs.obs['sample_id'].isin(removed_samples)].copy()

In [43]:
# Subset to CAR-T and endoT
adata_cart = adata_cart_crs[adata_cart_crs.obs['CAR-T_anno'] == 'CAR-T'].copy()
adata_endoT = adata_cart_crs[adata_cart_crs.obs['CAR-T_anno'] == 'endoT'].copy()

## Functional scoring genes
---
Fig. S3G

In [44]:
# Define gene lists to score T cell functional states
gene_lists = {
    'Proliferation': ['MKI67', 'STMN1', 'TOP2A', 'IL2RA', 'CD38'],
    'Cytotoxicity': ['GZMB', 'PRF1', 'NKG7', 'IFNG'],
    'Exhaustion': ['PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'TOX'],
    'Memory': ['IL7R', 'CCR7', 'SELL', 'TCF7', 'LEF1']
}

In [ ]:
# Regular gene-expression dotplots
stage_order = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']

def validate_gene_lists(adata, glists, object_name):
    missing = {
        state: [gene for gene in genes if gene not in adata.var_names]
        for state, genes in glists.items()
    }
    missing = {state: genes for state, genes in missing.items() if genes}
    if missing:
        raise ValueError(f"Missing genes in {object_name}: {missing}")

validate_gene_lists(adata_cart, gene_lists, 'CAR-T')
validate_gene_lists(adata_endoT, gene_lists, 'endoT')
cart_gene_lists = gene_lists
endoT_gene_lists = gene_lists

# CAR-T
dp_cart = sc.pl.dotplot(
    adata_cart,
    var_names=cart_gene_lists,
    groupby='stage',
    categories_order=stage_order,
    use_raw=False,
    standard_scale='var',
    cmap='OrRd',
    return_fig=True
)
dp_cart.savefig("dotplot_gene_expression_CAR-T.pdf", bbox_inches="tight")

# endoT
dp_endoT = sc.pl.dotplot(
    adata_endoT,
    var_names=endoT_gene_lists,
    groupby='stage',
    categories_order=stage_order,
    use_raw=False,
    standard_scale='var',
    cmap='OrRd',
    return_fig=True
)
dp_endoT.savefig("dotplot_gene_expression_endoT.pdf", bbox_inches="tight")


## Functional scoring

In [46]:
# Define scoring function using sc.tl.score_genes()

def score_sample_average(adata, gene_lists, sample_col='sample_id', group_col='dataset_group'):
    # work on a copy to avoid modifying the original AnnData obs
    ad = adata.copy()

    score_cols = []
    for state, genes in gene_lists.items():
        missing_genes = [gene for gene in genes if gene not in ad.var_names]
        if missing_genes:
            raise ValueError(f"Missing genes for {state}: {missing_genes}")

        score_name = f"{state}_score"

        sc.tl.score_genes(
            ad,
            gene_list=genes,
            score_name=score_name,
            use_raw=False
        )
        score_cols.append(score_name)

    # aggregate to sample level (mean score across cells)
    agg_cols = [sample_col] + score_cols
    if group_col in ad.obs.columns:
        agg_cols.append(group_col)

    df = ad.obs[agg_cols].copy()
    df_scores = df.groupby(sample_col, observed=True)[score_cols].mean()

    # rename columns back to state names
    rename_map = {f"{state}_score": state for state in gene_lists.keys()}
    df_scores = df_scores.rename(columns=rename_map)

    # optionally add group label per sample
    if group_col in ad.obs.columns:
        sample_group = ad.obs.groupby(sample_col, observed=True)[group_col].first()
        df_scores[group_col] = sample_group

    # reset index to retain the sample_col as a column
    return df_scores.reset_index()

In [47]:
# Run sample-average functional-state scoring for both cell types
cart_scores = score_sample_average(
    adata_cart,
    gene_lists,
    sample_col='sample_id',
    group_col='stage'
)

endoT_scores = score_sample_average(
    adata_endoT,
    gene_lists,
    sample_col='sample_id',
    group_col='stage'
)

# Keep stage order consistent
stage_order = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']
cart_scores['stage'] = pd.Categorical(cart_scores['stage'], categories=stage_order, ordered=True)
endoT_scores['stage'] = pd.Categorical(endoT_scores['stage'], categories=stage_order, ordered=True)

# Add cell-type label and merge
cart_scores['CAR-T_anno'] = 'CAR-T'
endoT_scores['CAR-T_anno'] = 'endoT'

## Radial plot
---
Fig. 3C, Table S4c

In [48]:
# Plotting data preparation

# Normalize the scores for heterogeneity, rather than absolute magnitudes

score_cols = ['Proliferation', 'Cytotoxicity', 'Exhaustion', 'Memory']

# objects to normalize and export
score_tables = {
    'cart': {
        'df': cart_scores,
        'id_cols': ['sample_id', 'stage'],
        'output': 'CAR-T_sample_average_scores_normalized.csv'
    },
    'endoT': {
        'df': endoT_scores,
        'id_cols': ['sample_id', 'stage'],
        'output': 'endoT_sample_average_scores_normalized.csv'
    }
}

for cell_type, config in score_tables.items():
    # create a normalized copy
    df_norm = config['df'].copy()

    # min-max normalization per score column
    for col in score_cols:
        min_val = df_norm[col].min()
        max_val = df_norm[col].max()

        # avoid division by zero
        if max_val - min_val > 0:
            df_norm[col] = (df_norm[col] - min_val) / (max_val - min_val)
        else:
            df_norm[col] = 0.0

    # row normalization so that functional-state scores sum to 1 within each sample
    df_norm['total_weight'] = df_norm[score_cols].sum(axis=1)
    df_norm = df_norm[df_norm['total_weight'] > 0].copy()

    for col in score_cols:
        df_norm[col] = df_norm[col] / df_norm['total_weight']

    # store normalized dataframe for downstream use
    config['df_norm'] = df_norm

    # export normalized scores (Table S4c)
    df_norm[config['id_cols'] + score_cols].to_csv(
        config['output'],
        index=False
    )

In [49]:
# Project transformed scores onto 2D space for visualization

for cell_type, config in score_tables.items():
    df_norm = config['df_norm']

    df_norm['x'] = df_norm['Exhaustion'] - df_norm['Memory']
    df_norm['y'] = df_norm['Proliferation'] - df_norm['Cytotoxicity']

df_cart_norm = score_tables['cart']['df_norm']
df_endoT_norm = score_tables['endoT']['df_norm']

In [52]:
# plotting preparation

import matplotlib.patches as mpatches

point_colors = {
    'CAR-T_CRS_before': "#284AC8",
    'CAR-T_CRS_pro': "#AC2C2C",
    'CAR-T_CRS_con': "#28A42E"
}

sector_colors = {
    'Proliferation': "#6d6dfc",
    'Cytotoxicity': "#56c756",
    'Exhaustion': "#ff6a6a",
    'Memory': "#ffd650"
}

time_order = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']


# functions
def plot_radial(df, adata, time_col, output,
                sectors,
                radius,
                rim_width=0.01,
                size_min=20,
                size_max=800):

    df = df.copy()

    # patient id
    if 'sample_id' in df.columns:
        df['patient_id'] = df['sample_id'].map(lambda x: str(x).rsplit('_', 1)[0])
    else:
        df['patient_id'] = df.index.map(lambda x: str(x).rsplit('_', 1)[0])

    # cell counts
    sample_counts = adata.obs['sample_id'].value_counts()

    if 'sample_id' in df.columns:
        df['cell_count'] = df['sample_id'].map(lambda x: sample_counts.get(x, 10))
    else:
        df['cell_count'] = df.index.map(lambda x: sample_counts.get(x, 10))

    df['cell_count'] = pd.to_numeric(df['cell_count'], errors='coerce').fillna(10).astype(int)

    # size scaling
    min_count, max_count = df['cell_count'].min(), df['cell_count'].max()

    if max_count > min_count:
        df['plot_size'] = np.interp(
            df['cell_count'],
            (min_count, max_count),
            (size_min, size_max)
        )
    else:
        df['plot_size'] = (size_min + size_max) / 2

    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)

    # sectors (dataset-specific)
    for name, start, end, color in sectors:
        ax.add_patch(
            mpatches.Wedge(
                center=(0, 0),
                r=radius,
                theta1=start,
                theta2=end,
                width=rim_width,
                color=color,
                alpha=1.0
            )
        )

    # trajectory ordering
    df[time_col] = pd.Categorical(df[time_col], categories=time_order, ordered=True)

    for pid, group in df.groupby('patient_id'):
        if len(group) > 1:
            group = group.copy()
            group['sort_val'] = group[time_col].map(lambda x: time_order.index(x))
            group = group.sort_values('sort_val')

            ax.plot(group['x'], group['y'],
                    color='black', alpha=0.7, linewidth=2, zorder=3)

    # points
    for g in time_order:
        sub = df[df[time_col] == g]
        if len(sub) > 0:
            ax.scatter(
                sub['x'], sub['y'],
                s=sub['plot_size'],
                color=point_colors[g],
                edgecolors='white',
                linewidth=1,
                alpha=1,
                zorder=4
            )

    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    plt.savefig(output, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    return df

In [ ]:
# plot cart
sectors_cart = [
    ('Exhaustion', -43, 43, sector_colors['Exhaustion']),
    ('Proliferation', 47, 133, sector_colors['Proliferation']),
    ('Memory', 137, 223, sector_colors['Memory']),
    ('Cytotoxicity', 227, 313, sector_colors['Cytotoxicity'])
]

df_cart_norm = plot_radial(
    df=df_cart_norm,
    adata=adata_cart,
    time_col='stage',
    output='CAR-T_functional_states.pdf',
    sectors=sectors_cart,
    radius=0.8
)

In [ ]:
# plot endoT
sectors_endoT = [
    ('Exhaustion', -42.5, 42.5, sector_colors['Exhaustion']),
    ('Proliferation', 47.5, 132.5, sector_colors['Proliferation']),
    ('Memory', 137.5, 222.5, sector_colors['Memory']),
    ('Cytotoxicity', 227.5, 312.5, sector_colors['Cytotoxicity'])
]

df_endoT_norm = plot_radial(
    df=df_endoT_norm,
    adata=adata_endoT,
    time_col='stage',
    output='CAR-T_functional_states.pdf',
    sectors=sectors_endoT,
    radius=1.0
)

# 8. CD4 and CD8 CAR-T functional scoring
---
Fig. S3E

In [56]:
# Subset to only CAR-T cells
adata_cart = adata_t[adata_t.obs['CAR-T_anno'] == 'CAR-T'].copy()

# remove No_CRS samples
adata_cart = adata_cart[adata_cart.obs['severity'] != 'No_CRS'].copy()

In [57]:
# Subset CAR-T data into CD4 and CD8 AnnData objects
adata_cart_cd4 = adata_cart[adata_cart.obs['t_anno'].astype(str).str.contains('CD4')].copy()
adata_cart_cd8 = adata_cart[adata_cart.obs['t_anno'].astype(str).str.contains('CD8')].copy()

In [58]:
# Define gene lists to score T cell functional states
gene_lists = {
    'Proliferation': ['MKI67', 'STMN1', 'TOP2A', 'IL2RA', 'CD38'],
    'Cytotoxicity': ['GZMB', 'PRF1', 'NKG7', 'IFNG'],
    'Exhaustion': ['PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'TOX'],
    'Memory': ['IL7R', 'CCR7', 'SELL', 'TCF7', 'LEF1']
}

In [59]:
# Define scoring function using sc.tl.score_genes()

def score_sample_average(adata, gene_lists, sample_col='sample_id', group_col='dataset_group'):
    # work on a copy to avoid modifying the original AnnData obs
    ad = adata.copy()

    score_cols = []
    for state, genes in gene_lists.items():
        missing_genes = [gene for gene in genes if gene not in ad.var_names]
        if missing_genes:
            raise ValueError(f"Missing genes for {state}: {missing_genes}")

        score_name = f"{state}_score"

        sc.tl.score_genes(
            ad,
            gene_list=genes,
            score_name=score_name,
            use_raw=False
        )
        score_cols.append(score_name)

    # aggregate to sample level (mean score across cells)
    agg_cols = [sample_col] + score_cols
    if group_col in ad.obs.columns:
        agg_cols.append(group_col)

    df = ad.obs[agg_cols].copy()
    df_scores = df.groupby(sample_col, observed=True)[score_cols].mean()

    # rename columns back to state names
    rename_map = {f"{state}_score": state for state in gene_lists.keys()}
    df_scores = df_scores.rename(columns=rename_map)

    # optionally add group label per sample
    if group_col in ad.obs.columns:
        sample_group = ad.obs.groupby(sample_col, observed=True)[group_col].first()
        df_scores[group_col] = sample_group

    # reset index to retain the sample_col as a column
    return df_scores.reset_index()

In [60]:
# Run sample-average functional-state scoring for both cell types
cd4_scores = score_sample_average(
    adata_cart_cd4,
    gene_lists,
    sample_col='sample_id',
    group_col='stage'
)

cd8_scores = score_sample_average(
    adata_cart_cd8,
    gene_lists,
    sample_col='sample_id',
    group_col='stage'
)

# Keep stage order consistent
stage_order = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']
cd4_scores['stage'] = pd.Categorical(cd4_scores['stage'], categories=stage_order, ordered=True)
cd8_scores['stage'] = pd.Categorical(cd8_scores['stage'], categories=stage_order, ordered=True)

# Add cell-type label and merge
cd4_scores['CAR-T_anno'] = 'CAR-T'
cd8_scores['CAR-T_anno'] = 'CAR-T'

In [61]:
# Plotting data preparation

# Normalize the scores for heterogeneity, rather than absolute magnitudes

score_cols = ['Proliferation', 'Cytotoxicity', 'Exhaustion', 'Memory']

# objects to normalize and export
score_tables = {
    'cd4': {
        'df': cd4_scores,
        'id_cols': ['sample_id', 'stage'],
        'output': 'CD4_CAR-T_sample_average_scores_normalized.csv'
    },
    'cd8': {
        'df': cd8_scores,
        'id_cols': ['sample_id', 'stage'],
        'output': 'CD8_CAR-T_sample_average_scores_normalized.csv'
    }
}

for cell_type, config in score_tables.items():
    # create a normalized copy
    df_norm = config['df'].copy()

    # min-max normalization per score column
    for col in score_cols:
        min_val = df_norm[col].min()
        max_val = df_norm[col].max()

        # avoid division by zero
        if max_val - min_val > 0:
            df_norm[col] = (df_norm[col] - min_val) / (max_val - min_val)
        else:
            df_norm[col] = 0.0

    # row normalization so that functional-state scores sum to 1 within each sample
    df_norm['total_weight'] = df_norm[score_cols].sum(axis=1)
    df_norm = df_norm[df_norm['total_weight'] > 0].copy()

    for col in score_cols:
        df_norm[col] = df_norm[col] / df_norm['total_weight']

    # store normalized dataframe for downstream use
    config['df_norm'] = df_norm

    # export normalized scores
    df_norm[config['id_cols'] + score_cols].to_csv(
        config['output'],
        index=False
    )

In [62]:
# Project transformed scores onto 2D space for visualization

for cell_type, config in score_tables.items():
    df_norm = config['df_norm']

    df_norm['x'] = df_norm['Exhaustion'] - df_norm['Memory']
    df_norm['y'] = df_norm['Proliferation'] - df_norm['Cytotoxicity']

df_cd4_norm = score_tables['cd4']['df_norm']
df_cd8_norm = score_tables['cd8']['df_norm']

In [ ]:
# Plot CD4 and CD8 CAR-T radial plots (Fig. S3E)

import matplotlib.patches as mpatches

# define time order
time_order = ['CAR-T_CRS_before', 'CAR-T_CRS_pro', 'CAR-T_CRS_con']

# Colors for 3 time points
point_colors = {
    'CAR-T_CRS_before': "#284AC8",       
    'CAR-T_CRS_pro': "#AC2C2C",  
    'CAR-T_CRS_con': "#28A42E" 
}

# Colors of the 4 sectors
sector_colors = {
    'Proliferation': "#6d6dfc", 
    'Cytotoxicity': "#56c756", 
    'Exhaustion': "#ff6a6a", 
    'Memory': "#ffd650"
}

# Objects to plot
plot_tables = {
    'cd4': {
        'df': df_cd4_norm,
        'adata': adata_cart_cd4,
        'time_col': 'stage',
        'output': 'cd4_functional_states.pdf'
    },
    'cd8': {
        'df': df_cd8_norm,
        'adata': adata_cart_cd8,
        'time_col': 'stage',
        'output': 'cd8_functional_states.pdf'
    }
}

for cell_type, config in plot_tables.items():

    df_norm = config['df'].copy()
    adata_obj = config['adata']
    time_col = config['time_col']

    # extract patient id
    if 'sample_id' in df_norm.columns:
        df_norm['patient_id'] = df_norm['sample_id'].map(lambda x: str(x).rsplit('_', 1)[0])
    else:
        df_norm['patient_id'] = df_norm.index.map(lambda x: str(x).rsplit('_', 1)[0])

    # map sample id with cell counts, which will be used for plotting dot sizes
    sample_counts = adata_obj.obs['sample_id'].value_counts()

    if 'sample_id' in df_norm.columns:
        df_norm['cell_count'] = df_norm['sample_id'].map(lambda x: sample_counts.get(x, 10))
    else:
        df_norm['cell_count'] = df_norm.index.map(lambda x: sample_counts.get(x, 10))

    df_norm['cell_count'] = pd.to_numeric(df_norm['cell_count'], errors='coerce').fillna(10).astype(int)

    # normalize sizes for plotting
    min_size = 20
    max_size = 800
    min_count = df_norm['cell_count'].min()
    max_count = df_norm['cell_count'].max()

    # Linear interpolation for size, with a constant-size fallback.
    if max_count > min_count:
        df_norm['plot_size'] = np.interp(
            df_norm['cell_count'],
            (min_count, max_count),
            (min_size, max_size)
        )
    else:
        df_norm['plot_size'] = (min_size + max_size) / 2

    fig, ax = plt.subplots(figsize=(10,10), dpi=300)

    # Sectors
    rim_width = 0.01
    radius = 0.8
    sectors = [
        ('Exhaustion', -43, 43, sector_colors['Exhaustion']),
        ('Proliferation', 47, 133, sector_colors['Proliferation']),
        ('Memory', 137, 223, sector_colors['Memory']),
        ('Cytotoxicity', 227, 313, sector_colors['Cytotoxicity'])
    ]

    for name, start, end, color in sectors:
        # draw the rim
        wedge = mpatches.Wedge(
            center=(0,0),
            r=radius,
            theta1=start,
            theta2=end,
            width=rim_width,
            color=color,
            alpha=1.0
        )
        ax.add_patch(wedge)

        # add label outside the rim, calculate the middle angle
        mid_angle_rad = np.deg2rad((start + end) / 2)
        lbl_x = (radius + 0.1) * np.cos(mid_angle_rad)
        lbl_y = (radius + 0.1) * np.sin(mid_angle_rad)

    # draw patient trajectories
    # convert stage to ordered categorical for sorting
    df_norm[time_col] = pd.Categorical(
        df_norm[time_col],
        categories=time_order,
        ordered=True
    )

    grouped = df_norm.groupby('patient_id')

    for pid, group in grouped:
        if len(group) > 1:
            # sort by time order
            group = group.copy()
            group['sort_val'] = group[time_col].map(lambda x: time_order.index(x) if x in time_order else 99)
            group = group.sort_values('sort_val')

            # draw black polyline
            ax.plot(group['x'], group['y'], color='black', alpha=0.7, linewidth=2, zorder=3)

    # points
    # plot each group separately to assign colors
    for group_name in time_order:
        subset = df_norm[df_norm[time_col] == group_name]

        if len(subset) > 0:
            ax.scatter(
                subset['x'], subset['y'],
                color=point_colors.get(group_name, 'grey'),
                s=subset['plot_size'], 
                edgecolors='white',
                alpha=1.0,
                linewidth=1.0,
                label=group_name,
                zorder=4
            )

    ax.set_xlim(-1.0, 1.0)
    ax.set_ylim(-1.0, 1.0)
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()

    # save
    plt.savefig(config['output'], dpi=300, bbox_inches='tight')

    plt.show()
    plt.close()

    # store modified dataframe back for downstream use
    plot_tables[cell_type]['df'] = df_norm

df_cd4_norm = plot_tables['cd4']['df']
df_cd8_norm = plot_tables['cd8']['df']